In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from pathlib import Path

# ============================================================
# Ejemplo visual: operador singular de Cauchy sobre una
# star-like curve Gamma = union Gamma_k.
#
# Usamos dos tipos de ramas:
#
# 1) Ramas cuspídeas:
#       gamma_k(x) = 1 / (x - i c_k),     x in [a,b]
#
# 2) Ramas tipo rayo:
#       gamma_k(x) = exp(i beta_k) / x,   x in [a,b]
#
# Todas se aproximan cerca del punto común 0 cuando x -> infinito.
#
# La integral de Cauchy se discretiza sobre todas las ramas:
#
#     (S_Gamma f)(t_j) ≈ 1/(pi i) sum_{k != j}
#          f(t_k) dgamma_k / (t_k - t_j).
#
# Esto es una visualización numérica, no una prueba rigurosa.
# ============================================================

# -----------------------------
# Parámetros generales
# -----------------------------
a, b = 0.08, 9.0
N_branch = 260
x = np.linspace(a, b, N_branch)
dx = x[1] - x[0]

branches = []

# -----------------------------
# Ramas cuspídeas: gamma(x)=1/(x - i c)
# -----------------------------
c_values = [-1.4, -0.65, 0.65, 1.4]

for c in c_values:
    gamma = 1.0 / (x - 1j*c)
    gamma_prime = -1.0 / (x - 1j*c)**2
    branches.append({
        "kind": "cusp",
        "param": c,
        "gamma": gamma,
        "gamma_prime": gamma_prime
    })

# -----------------------------
# Ramas tipo rayo: gamma(x)=exp(i beta)/x
# Las ponemos en el semiplano izquierdo para dar aspecto star-like.
# -----------------------------
beta_values = [2.25, 2.75, 3.35, 3.85]

for beta in beta_values:
    gamma = np.exp(1j*beta) / x
    gamma_prime = -np.exp(1j*beta) / x**2
    branches.append({
        "kind": "ray",
        "param": beta,
        "gamma": gamma,
        "gamma_prime": gamma_prime
    })

# Unimos todos los puntos de la curva
Gamma = np.concatenate([br["gamma"] for br in branches])
Gamma_prime = np.concatenate([br["gamma_prime"] for br in branches])
Xparam = np.concatenate([x for _ in branches])
branch_id = np.concatenate([
    np.full(N_branch, k) for k in range(len(branches))
])

M = len(Gamma)

# -----------------------------
# Función f sobre la star-like curve
# f depende del parámetro x y de la rama.
# -----------------------------
#f = np.exp(-0.28 * Xparam) * np.exp(1j * (1.7 * Xparam + 0.7 * branch_id))
f = np.exp(-0.20 * Xparam) * (np.cos(3.0 * Xparam) + 1j*np.sin(1.5 * Xparam))

# -----------------------------
# Aproximación discreta del operador S_Gamma f
# -----------------------------
Sf = np.zeros(M, dtype=complex)

for j in range(M):
    denom = Gamma - Gamma[j]
    mask = np.ones(M, dtype=bool)

    # Valor principal: quitar el punto singular exacto
    mask[j] = False

    # Para estabilizar la visualización cerca de puntos muy próximos,
    # también quitamos vecinos demasiado cercanos.
    mask &= np.abs(denom) > 1e-5

    Sf[j] = (1.0 / (np.pi * 1j)) * np.sum(
        f[mask] * Gamma_prime[mask] * dx / denom[mask]
    )

# -----------------------------
# Archivos de salida
# -----------------------------
outdir = Path(".")
png_path = outdir / "cauchy_operator_star_like_curve_static.png"
gif_path = outdir / "cauchy_operator_star_like_curve.gif"

# -----------------------------
# Figura estática
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax in axes:
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.axis("equal")
    ax.grid(True)

# Curva
for br in branches:
    axes[0].plot(br["gamma"].real, br["gamma"].imag, linewidth=1.8)

axes[0].scatter([0], [0], s=45, marker="x", label="punto común 0")
axes[0].set_title(r"Star-like curve $\Gamma=\bigcup_k \Gamma_k$")
axes[0].legend()

# f sobre Gamma
sc1 = axes[1].scatter(Gamma.real, Gamma.imag, c=np.real(f), s=8)
axes[1].set_title(r"$f$ sobre $\Gamma$: color = Re$(f)$")
fig.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.04)

# S_Gamma f sobre Gamma
sc2 = axes[2].scatter(Gamma.real, Gamma.imag, c=np.real(Sf), s=8)
axes[2].set_title(r"$S_\Gamma f$ sobre $\Gamma$: color = Re$(S_\Gamma f)$")
fig.colorbar(sc2, ax=axes[2], fraction=0.046, pad=0.04)

fig.tight_layout()
fig.savefig(png_path, dpi=180)
plt.close(fig)

# -----------------------------
# GIF animado
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax in axes:
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.axis("equal")
    ax.grid(True)

axes[0].set_title(r"1. Star-like curve $\Gamma$")
axes[1].set_title(r"2. Función $f$ sobre $\Gamma$")
axes[2].set_title(r"3. Imagen $S_\Gamma f$")

xmin, xmax = Gamma.real.min(), Gamma.real.max()
ymin, ymax = Gamma.imag.min(), Gamma.imag.max()
padx = 0.08 * (xmax - xmin)
pady = 0.08 * (ymax - ymin)

for ax in axes:
    ax.set_xlim(xmin - padx, xmax + padx)
    ax.set_ylim(ymin - pady, ymax + pady)

# Objetos por rama para la curva
curve_lines = []
for _ in branches:
    line, = axes[0].plot([], [], linewidth=1.8)
    curve_lines.append(line)

sc_f = axes[1].scatter([], [], c=[], s=10, vmin=np.real(f).min(), vmax=np.real(f).max())
sc_Sf = axes[2].scatter([], [], c=[], s=10, vmin=np.real(Sf).min(), vmax=np.real(Sf).max())

cb1 = fig.colorbar(sc_f, ax=axes[1], fraction=0.046, pad=0.04)
cb1.set_label(r"Re$(f)$")
cb2 = fig.colorbar(sc_Sf, ax=axes[2], fraction=0.046, pad=0.04)
cb2.set_label(r"Re$(S_\Gamma f)$")

frames = 130

def update(frame):
    m = int(2 + (N_branch - 2) * frame / (frames - 1))

    # Curva progresiva por rama
    for k, br in enumerate(branches):
        curve_lines[k].set_data(br["gamma"].real[:m], br["gamma"].imag[:m])

    # Puntos acumulados rama por rama
    selected = []
    for k in range(len(branches)):
        start = k * N_branch
        selected.extend(range(start, start + m))

    selected = np.array(selected)

    sc_f.set_offsets(np.column_stack([Gamma.real[selected], Gamma.imag[selected]]))
    sc_f.set_array(np.real(f[selected]))

    sc_Sf.set_offsets(np.column_stack([Gamma.real[selected], Gamma.imag[selected]]))
    sc_Sf.set_array(np.real(Sf[selected]))

    return curve_lines + [sc_f, sc_Sf]

anim = FuncAnimation(fig, update, frames=frames, interval=60, blit=False)
anim.save(gif_path, writer=PillowWriter(fps=18))
plt.close(fig)

gif_path, png_path


(PosixPath('cauchy_operator_star_like_curve.gif'),
 PosixPath('cauchy_operator_star_like_curve_static.png'))

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from pathlib import Path

# ============================================================
# Visualización corregida:
#
# 1) Gráfico de f como señal sobre el parámetro x.
# 2) Gráfico geométrico de la curva star-like Gamma.
# 3) Gráfico del operador de Cauchy S_Gamma aplicado a f.
#
# El tercer gráfico NO es Gamma coloreada; es la imagen transformada
# S_Gamma f, graficada como una señal nueva.
# ============================================================

# Parámetros
a, b = 0.08, 9.0
N_branch = 260
x = np.linspace(a, b, N_branch)
dx = x[1] - x[0]

branches = []

# Ramas cuspídeas
c_values = [-1.4, -0.65, 0.65, 1.4]
for c in c_values:
    gamma = 1.0 / (x - 1j*c)
    gamma_prime = -1.0 / (x - 1j*c)**2
    branches.append(("cusp", c, gamma, gamma_prime))

# Ramas tipo rayo
beta_values = [2.25, 2.75, 3.35, 3.85]
for beta in beta_values:
    gamma = np.exp(1j*beta) / x
    gamma_prime = -np.exp(1j*beta) / x**2
    branches.append(("ray", beta, gamma, gamma_prime))

# Unir todos los puntos
Gamma = np.concatenate([br[2] for br in branches])
Gamma_prime = np.concatenate([br[3] for br in branches])
Xparam = np.concatenate([x for _ in branches])
branch_id = np.concatenate([np.full(N_branch, k) for k in range(len(branches))])
M = len(Gamma)

# Nueva función f: real, oscilatoria y con decaimiento
f = np.exp(-0.007 * Xparam) * np.sin(3.0 * Xparam + 0.45 * branch_id)
#f = np.exp(-0.30 * Xparam)
#f = np.exp(-0.25 * Xparam) * np.sin(2.0 * Xparam + 0.8 * branch_id)
#f = np.exp(-4.5 * Xparam) * np.exp(1j * (2.0 * Xparam + 4.5 * branch_id))
# Operador singular de Cauchy
Sf = np.zeros(M, dtype=complex)

for j in range(M):
    denom = Gamma - Gamma[j]
    mask = np.ones(M, dtype=bool)
    mask[j] = False
    mask &= np.abs(denom) > 1e-5

    Sf[j] = (1.0 / (np.pi * 1j)) * np.sum(
        f[mask] * Gamma_prime[mask] * dx / denom[mask]
    )

# Para el tercer gráfico usamos una señal real asociada al operador:
# Re(S_Gamma f). También se podría usar Im(S_Gamma f) o |S_Gamma f|.
Sf_real = np.real(Sf)

# Salidas
outdir = Path(".")
png_path = outdir / "f_gamma_Sf_three_graphs_static.png"
gif_path = outdir / "f_gamma_Sf_three_graphs.gif"

# -----------------------------
# Figura estática
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# 1) Gráfico de f
for k in range(len(branches)):
    start = k * N_branch
    end = start + N_branch
    axes[0].plot(x, f[start:end], linewidth=1.4, alpha=0.9)

axes[0].set_title(r"1. Gráfico de $f(\gamma_k(x))$")
axes[0].set_xlabel(r"Parámetro $x$")
axes[0].set_ylabel(r"$f$")
axes[0].grid(True)

# 2) Gráfico de Gamma
for _, _, gamma, _ in branches:
    axes[1].plot(gamma.real, gamma.imag, linewidth=1.8)
axes[1].scatter([0], [0], marker="x", s=60, label="punto común 0")
axes[1].set_title(r"2. Curva star-like $\Gamma$")
axes[1].set_xlabel("Re")
axes[1].set_ylabel("Im")
axes[1].axis("equal")
axes[1].grid(True)
axes[1].legend()

# 3) Gráfico del operador S_Gamma f
for k in range(len(branches)):
    start = k * N_branch
    end = start + N_branch
    axes[2].plot(x, Sf_real[start:end], linewidth=1.4, alpha=0.9)

axes[2].set_title(r"3. Gráfico de $S_\Gamma f$")
axes[2].set_xlabel(r"Parámetro $x$")
axes[2].set_ylabel(r"Re$(S_\Gamma f)$")
axes[2].grid(True)

fig.tight_layout()
fig.savefig(png_path, dpi=180)
plt.close(fig)

# -----------------------------
# GIF animado
# -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].set_title(r"1. Gráfico de $f(\gamma_k(x))$")
axes[0].set_xlabel(r"Parámetro $x$")
axes[0].set_ylabel(r"$f$")
axes[0].grid(True)
axes[0].set_xlim(a, b)
axes[0].set_ylim(f.min()*1.1, f.max()*1.1)

axes[1].set_title(r"2. Curva star-like $\Gamma$")
axes[1].set_xlabel("Re")
axes[1].set_ylabel("Im")
axes[1].axis("equal")
axes[1].grid(True)

xmin, xmax = Gamma.real.min(), Gamma.real.max()
ymin, ymax = Gamma.imag.min(), Gamma.imag.max()
padx = 0.08 * (xmax - xmin)
pady = 0.08 * (ymax - ymin)
axes[1].set_xlim(xmin - padx, xmax + padx)
axes[1].set_ylim(ymin - pady, ymax + pady)

axes[2].set_title(r"3. Gráfico de $S_\Gamma f$")
axes[2].set_xlabel(r"Parámetro $x$")
axes[2].set_ylabel(r"Re$(S_\Gamma f)$")
axes[2].grid(True)
axes[2].set_xlim(a, b)
axes[2].set_ylim(Sf_real.min()*1.1, Sf_real.max()*1.1)

# Objetos de animación
f_lines = []
gamma_lines = []
Sf_lines = []

for k in range(len(branches)):
    lf, = axes[0].plot([], [], linewidth=1.4)
    lg, = axes[1].plot([], [], linewidth=1.8)
    ls, = axes[2].plot([], [], linewidth=1.4)
    f_lines.append(lf)
    gamma_lines.append(lg)
    Sf_lines.append(ls)

axes[1].scatter([0], [0], marker="x", s=60)

frames = 130

def update(frame):
    m = int(2 + (N_branch - 2) * frame / (frames - 1))

    for k, (_, _, gamma, _) in enumerate(branches):
        start = k * N_branch
        end = start + m

        f_lines[k].set_data(x[:m], f[start:end])
        gamma_lines[k].set_data(gamma.real[:m], gamma.imag[:m])
        Sf_lines[k].set_data(x[:m], Sf_real[start:end])

    return f_lines + gamma_lines + Sf_lines

anim = FuncAnimation(fig, update, frames=frames, interval=60, blit=False)
anim.save(gif_path, writer=PillowWriter(fps=18))
plt.close(fig)

gif_path, png_path


(PosixPath('f_gamma_Sf_three_graphs.gif'),
 PosixPath('f_gamma_Sf_three_graphs_static.png'))